# [BANC QUICK TEST] FlyWire Error Analysis: False Synapses
**Quick end-to-end pipeline validation using the BANC dataset.**
# Cell 3 sets DATASET_NAME to BANC. Edit it to change dataset.
---
### How to run
1. Attach `flywire-codebase` and `flywire-all-datasets` Kaggle datasets.
**Completes in ~30-45 min on Kaggle (2 rates x 1 seed + candidate gen).**
---
### Kaggle Dataset Paths
- Codebase : `/kaggle/input/datasets/jeet7771/flywire-codebase`
- Data      : `/kaggle/input/datasets/jeet7771/flywire-all-datasets`


In [ ]:
# Cell 1: Environment Setup & sys.path
import os, sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')
KAGGLE_CODEBASE_PATH = Path('/kaggle/input/datasets/jeet7771/flywire-codebase')
KAGGLE_DATA_PATH     = Path('/kaggle/input/datasets/jeet7771/flywire-all-datasets')

if IS_KAGGLE:
    if not KAGGLE_CODEBASE_PATH.exists():
        raise FileNotFoundError(f'Codebase not at {KAGGLE_CODEBASE_PATH}')
    sys.path.insert(0, str(KAGGLE_CODEBASE_PATH))
    if not KAGGLE_DATA_PATH.exists():
        raise FileNotFoundError(f'Data not at {KAGGLE_DATA_PATH}')
    print(f'[OK] Codebase: {KAGGLE_CODEBASE_PATH}')
    print(f'[OK] Data:     {KAGGLE_DATA_PATH}')
else:
    REPO_ROOT = Path(os.getcwd())
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    print(f'[OK] Local: {REPO_ROOT}')
print(f'Environment: {"KAGGLE" if IS_KAGGLE else "LOCAL"}')

In [ ]:
# Cell 2: Framework Imports
import warnings
import pandas as pd
warnings.filterwarnings('ignore')

from core.experiment_runner import ExperimentRunner, ExperimentConfig
from modules.error_models import registry as error_registry
from modules.graph_analyses.analysis_registry import registry as analysis_registry
from modules.statistical_evaluation import StatisticalEvaluator
from core.export_manager import ExportManager
from modules.preprocessing import CandidateGenerator

print('All imports successful.')

In [ ]:
# ============================================================
# Cell 3: MINIMAL TEST CONFIG  (fast run: 2 rates, 2 seeds)
# ============================================================
DATASET_NAME = "BANC"

EXPERIMENT = {
    "metadata": {
        "experiment_name": f"BANC_FalseSynapses_{DATASET_NAME}",
        "author": "FlyWire Researcher",
        "description": "QUICK TEST — false synapses on BANC dataset (reduced rates/seeds for testing).",
    },
    "error": {
        "name": "false_synapses",
        "rates": [
            0.00,    # baseline (required)
            0.02,    # 2% — one perturbed rate
        ],
        "random_seeds": [1],
    },
    "analysis": [
        "basic_structure",
        "degree_distribution",
        "pagerank",
        "assortativity",
        # "centrality",  # HEAVY — keep disabled
        "connected_components",
        "reciprocity",
    ],
    "export": {
        "create_zip": True,
        "save_statistics": True,
    },
}

OUTPUT_ROOT = Path("results") / DATASET_NAME / EXPERIMENT["error"]["name"]
CACHE_DIR = Path("/kaggle/working/cache/false_synapses") if IS_KAGGLE else Path("0-demo test/false synapse/cache")
print(f'Config: {DATASET_NAME} | rates={len(EXPERIMENT["error"]["rates"])} | seeds={len(EXPERIMENT["error"]["random_seeds"])}')

In [ ]:
# Cell 4: Resolve Dataset Root
if IS_KAGGLE:
    DATASET_ROOT = str(KAGGLE_DATA_PATH)
else:
    DATASET_ROOT = '0-demodata' if DATASET_NAME.upper() == 'TEST' else 'research_data/raw'
CONFIGS_ROOT = str(KAGGLE_CODEBASE_PATH / 'configs') if IS_KAGGLE else 'configs'
print(f'DATASET_ROOT = {DATASET_ROOT}')

In [ ]:
# Cell 5: Verify Dataset
from core.dataset_registry import DatasetRegistry, DatasetRegistryError
try:
    reg = DatasetRegistry(configs_root=CONFIGS_ROOT, dataset_root=DATASET_ROOT)
    resolved_dir = reg.resolve_dataset_dir(DATASET_NAME, DATASET_ROOT)
    print(f'[OK] Dataset "{DATASET_NAME}" -> {resolved_dir}')
except DatasetRegistryError as e:
    raise FileNotFoundError(f'Cannot resolve "{DATASET_NAME}" in "{DATASET_ROOT}": {e}') from e

In [ ]:
# Cell 6: Verify Registries
err_model = EXPERIMENT['error']['name']
print(f'Error Models : {error_registry.list_names()}')
print(f'Analyses     : {analysis_registry.list_names()}')
assert err_model in error_registry.list_names(), f'{err_model} not registered'
missing = [a for a in EXPERIMENT['analysis'] if a not in analysis_registry.list_names()]
assert not missing, f'Missing analyses: {missing}'
print('[OK] All components ready.')

In [ ]:
# Cell 7: Candidate Generation (False Synapse — runs once, cached)
import time
from core.data_loader import load_dataset
from core.graph_builder import GraphBuilder
from modules.preprocessing import preprocess_graph

t_cand = time.perf_counter()
dataset = load_dataset(DATASET_NAME, DATASET_ROOT, configs_root=CONFIGS_ROOT)
graph = GraphBuilder().build(dataset)
prepared = preprocess_graph(graph, index_node_attrs=['top_region'], feature_config={
    'indegree': True, 'outdegree': True, 'pagerank': False,
    'reciprocal_ratio': False, 'hub_neighbor_count': False, 'two_hop_size': False,
})
print(f'Loaded: {prepared.metadata.node_count} nodes, {prepared.metadata.edge_count} edges')

CACHE_DIR.mkdir(parents=True, exist_ok=True)
cache_path = CACHE_DIR / f'candidates_{DATASET_NAME.lower()}.parquet'

if cache_path.exists():
    import polars as pl
    cand_table = pl.read_parquet(str(cache_path))
    print(f'Cached candidates: {len(cand_table):,}')
else:
    generator = CandidateGenerator(prepared)
    generator.generate(cache_path)
    import polars as pl
    cand_table = pl.read_parquet(str(cache_path))
    print(f'Generated {len(cand_table):,} candidates in {time.perf_counter()-t_cand:.1f}s')

del prepared, graph, dataset

In [ ]:
# Cell 8: Run Experiments (2 rates x 1 seed = 2 trials)
t_start = time.perf_counter()
runner = ExperimentRunner(analysis_registry, error_registry)
results_per_rate = {}

for err_rate in EXPERIMENT['error']['rates']:
    rate_str = f"{int(err_rate*100)}_percent"
    results_per_rate[err_rate] = []
    for trial, seed in enumerate(EXPERIMENT['error']['random_seeds'], 1):
        print(f'[{rate_str} | trial {trial}] seed={seed} ...', end=' ')
        trial_out = OUTPUT_ROOT / rate_str / f'trial_{trial:03d}'
        config = ExperimentConfig(
            dataset_name=DATASET_NAME,
            dataset_root=str(DATASET_ROOT),
            configs_root=CONFIGS_ROOT,
            error_model_name=err_model,
            error_model_config={'error_rate': err_rate, 'candidate_cache_path': str(cache_path)},
            analysis_names=EXPERIMENT['analysis'],
            preprocessing_config={'features': {'degree': True, 'synapse_counts': True}},
            seed=seed,
            output_root=str(trial_out) if EXPERIMENT['export']['save_statistics'] else None,
            create_zip=EXPERIMENT['export']['create_zip'],
            extra={'metadata': EXPERIMENT['metadata']},
        )
        res = runner.run(config)
        results_per_rate[err_rate].append(res)
        status = 'OK' if res.succeeded else 'FAIL'
        print(f'{status} ({res.runtime_seconds:.2f}s)')

print(f'\nAll trials done in {time.perf_counter()-t_start:.1f}s')

In [ ]:
# Cell 9: Statistical Evaluation
evaluator = StatisticalEvaluator()
aggregated_stats_by_rate = {}
baseline_runs = [r for r in results_per_rate.get(0.00, []) if r.succeeded]
print(f'Baseline runs: {len(baseline_runs)}')

for err_rate, run_results in results_per_rate.items():
    successful = [r for r in run_results if r.succeeded]
    if successful and err_rate > 0:
        eval_result = evaluator.evaluate(baseline_runs, successful)
        aggregated_stats_by_rate[err_rate] = eval_result
        print(f'  {err_rate*100:g}%: {len(successful)} trials evaluated')

print('Evaluation complete.')

In [ ]:
# Cell 10: Export Presentation
ExportManager().export_presentation(
    results_by_rate=aggregated_stats_by_rate,
    output_root=OUTPUT_ROOT,
    metadata=EXPERIMENT['metadata'],
)
print(f'Presentation -> {OUTPUT_ROOT / "presentation"}')

In [ ]:
# Cell 11: Quick Summary
print('=' * 50)
print('  BANC QUICK TEST - FALSE SYNAPSES')
print('=' * 50)
for err_rate in sorted(aggregated_stats_by_rate.keys()):
    ev = aggregated_stats_by_rate[err_rate]
    print(f'  Error Rate {err_rate*100:g}%')
    for a_name, metrics in ev.metrics.items():
        print(f'    {a_name}: {len(metrics)} metrics')
        for m_name, m_dict in list(metrics.items())[:3]:
            print(f'      {m_name}: mean={m_dict.mean:.4f} d={m_dict.effect_size:.4f}')
total_trials = sum(len(v) for v in results_per_rate.values())
print(f'  Trials: {total_trials} | Rates: {len(aggregated_stats_by_rate)}')
print(f'  Output: {OUTPUT_ROOT}')
print('=' * 50)